# Local Flip Compression Geometry (v1)

Legacy plot notebook. Reads `data/all_settings_master.tsv` and plots the historical `perp_over_x` metric.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, MaxNLocator
import pandas as pd


In [ ]:
NOTEBOOK_DIR = Path('/Users/bytedance/Documents/GitHub/demystifying-transformers/_NeurIPS_2026_/drawing/comp_analysis')
PAPER_ROOT = NOTEBOOK_DIR.parents[1]
DATA_PATH = NOTEBOOK_DIR / 'data' / 'all_settings_master.tsv'
OUT_DIR = PAPER_ROOT / 'figs' / 'comp_analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

KEEP_SETTINGS = ['awq_native', 'wanda_2_4', 'wanda_4_8', 'wanda_unstructured']
PLOT_ORDER = ['awq_native', 'wanda_unstructured', 'wanda_4_8', 'wanda_2_4']
COMPONENTS = ['block_out', 'attn_out', 'mlp_out']

DISPLAY_NAME = {
    'awq_native': 'Quantization',
    'wanda_2_4': '2:4',
    'wanda_4_8': '4:8',
    'wanda_unstructured': 'Unstructured',
}

STYLE_MAP = {
    'awq_native': {'color': '#4C78A8', 'marker': 'o'},
    'wanda_unstructured': {'color': '#222222', 'marker': 's'},
    'wanda_4_8': {'color': '#F58518', 'marker': '^'},
    'wanda_2_4': {'color': '#E45756', 'marker': 'D'},
}

PLOT_RC = {
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.8,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 10.5,
    'ytick.labelsize': 10.5,
    'legend.fontsize': 10,
    'font.family': 'DejaVu Sans',
}


In [ ]:
def load_local_rows():
    df = pd.read_csv(DATA_PATH, sep='	')
    if 'focus_layer' in df.columns:
        df = df[df['layer'] == df['focus_layer']].copy()
    for col in ('perp_over_x', 'perp_over_x_std'):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def exclude_edge_layers(df):
    layers = sorted(df['layer'].dropna().unique().tolist())
    if len(layers) < 3:
        return df
    return df[(df['layer'] >= layers[1]) & (df['layer'] <= layers[-2])].copy()


def nice_ylim(series, pad=0.12):
    s = pd.to_numeric(series, errors='coerce').dropna()
    if s.empty:
        return None
    hi = float(s.quantile(0.98))
    if hi <= 0:
        hi = float(s.max())
    return (0.0, hi * (1.0 + pad))


In [ ]:
df = load_local_rows()
print(f'Loaded: {DATA_PATH}')
print(f'Rows after local focus-layer filter: {len(df)}')
print('Settings:', sorted(df['setting'].dropna().unique().tolist()))
df.head()


In [ ]:
def plot_v1_perp_over_x(df):
    metric = 'perp_over_x'
    ylabel = r'$\|\Delta_{\perp}\| / \|x\|$'
    d_all = exclude_edge_layers(df[df['setting'].isin(KEEP_SETTINGS)].copy())
    saved = []

    for comp in COMPONENTS:
        d = d_all[d_all['component'] == comp].copy()
        if d.empty or d[metric].notna().sum() == 0:
            continue

        with plt.rc_context(PLOT_RC):
            fig, ax = plt.subplots(figsize=(10.4, 4.2), dpi=180, constrained_layout=True)
            valid_settings = [s for s in PLOT_ORDER if d[d['setting'] == s][metric].notna().any()]

            for setting in valid_settings:
                g = d[d['setting'] == setting].sort_values('layer')
                style = STYLE_MAP.get(setting, {'color': '#4C78A8', 'marker': 'o'})
                line = ax.plot(
                    g['layer'], g[metric],
                    color=style['color'], marker=style['marker'], linewidth=2.0,
                    markersize=5.0, markeredgewidth=0.8,
                    label=DISPLAY_NAME.get(setting, setting), zorder=3,
                )[0]
                std_col = f'{metric}_std'
                if std_col in g.columns and g[std_col].notna().any():
                    y = g[metric].astype(float)
                    ystd = g[std_col].fillna(0.0).astype(float)
                    ax.fill_between(
                        g['layer'], y - ystd, y + ystd,
                        color=line.get_color(), alpha=0.08, linewidth=0, zorder=1,
                    )

            layers = sorted(d['layer'].dropna().unique().tolist())
            if layers:
                ax.set_xlim(min(layers), max(layers))
                ax.xaxis.set_major_locator(MaxNLocator(nbins=min(10, len(layers)), integer=True))
            ax.set_xlabel('Layer')
            ax.set_ylabel(ylabel)
            ax.grid(True, color='#d0d0d0', linewidth=0.8, alpha=0.45)
            ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
            ylim = nice_ylim(d[metric])
            if ylim is not None:
                ax.set_ylim(*ylim)
            ax.legend(
                loc='upper center', bbox_to_anchor=(0.5, 0.99),
                ncol=min(4, len(valid_settings)), frameon=True,
                facecolor='white', edgecolor='#cfcfcf', borderpad=0.3,
                handlelength=2.0, columnspacing=1.2,
            )

            for suffix in ('pdf', 'png'):
                path = OUT_DIR / f'local_flip_compare_v1-{comp}-{metric}.{suffix}'
                fig.savefig(path, bbox_inches='tight', dpi=220 if suffix == 'png' else None)
                saved.append(path)

            if comp == 'attn_out':
                for suffix in ('pdf', 'png'):
                    compat = OUT_DIR / f'local_flip_compare-{comp}-{metric}.{suffix}'
                    fig.savefig(compat, bbox_inches='tight', dpi=220 if suffix == 'png' else None)
                    saved.append(compat)
            plt.show()
    return saved

saved = plot_v1_perp_over_x(df)
for path in saved:
    print(f'Saved: {path}')


In [ ]:
print('Figure directory:', OUT_DIR)
print('Generated v1 files:')
for path in sorted(OUT_DIR.glob('local_flip_compare_v1-*.pdf')):
    print(' ', path.name)
